<p>
<h1><b><center></center></b></h1>
<center><img src="https://drive.google.com/uc?id=1UJc1ci41G6ahJ7ProKvunUOIBcTXZ6ZG" align="center" width="550"></center>
</p>
<h1><b><center>Dinámica del Vuelo Espacial</center></b></h1>
<h2><b><center>Prof. Jorge I. Zuluaga</center></b></h1>
<h2><b><center>Assignment 2</center></b><h2>
<h3><b><center>Rendezvous in Mars</center></b><h3>
<h5><center><b>Assigned</b>: 31 de marzo de 2025</center><h5>
<h5><center><b>Submission</b>: 21 de abril de 2025</center><h5>

<hr/>
<b>Name</b>: Naireth Farak Román
<br/>
<b>Cédula</b>: 1102812285
<br/>
<b>Última actualización</b>: 21/04/2025 12:05 a.m
<hr/>

## Statement


**Procedure**:

**Una estación espacial** se encuentra alrededor de Marte en una órbita circular. Desde allí se quiere lanzar una **sonda con provisiones** que debe acoplarse con un **vehículo tripulado** que se encuentra en una órbitra mayor.  A este tipo de maniobras se las llama *rendezvous*.

En el momento en el que se decide realizar la maniobra (que asumiremos igual a $t=0$) el estado de la estación y del vehículo son:

- $r_\text{veh}$: [ 2316.794, -2907.198,  0.000] km
- $v_\text{veh}$: [ 3.651,  2.456,  0.000] km
- $r_\text{est}$: [ 11625.240, -4740.578,  0.000] km
- $v_\text{est}$: [ 0.696,  1.707, 0.000] km/s

La maniobra de Rendezvous debe comenzar en t = 3 horas con el lanzamiento del vehículo de provisiones desde el lugar en el que está la estación en ese momento.

Determine, mediante una búsqueda numérica, la velocidad con la que debe salir la sonda con provisiones desde la estación espacial para alcanzar el vehículo tripulado exactamente 200 minutos después. Draw and animate the maneuver.  

Haga un informe completo de la misión incluyendo, tiempos, posiciones y velocidades de los vehículos en los momentos críticos, cambios en la velocidad que debe sufrir la sonda con provisiones (para salir de la estación y para acoplarse al vehículo tripulado) y propiedades orbitales ($a, p, q, e, I, \omega, T$) de la estación, del vehículo y de la sonda con provisiones.

**NOTE**: Para la masa y el radio de Marte asuma como valores $R_\text{Marte} = 3393.149$ km y $M_\text{Marte} = 6.3902\times 10^{23}$ kg


## Conditions

Lea con atención está lista de condiciones que deben cumplirse al entregar la tarea para evitar sanciones en la calificación asignada:

  - La solución presentada debe ser estrictamente individual. Evite resolver la tarea en parejas o en grupos que puede conducir a códigos o soluciones idénticos o muy similares.
  - Los métodos y herramientas para resolver el problema deben ser los vistos en clase. El uso de herramientas diferentes puede ser una buena práctica en el mundo académico o laboral, pero en un curso puede también ser un indicio de un mal uso de las *asesorías* externa o del uso inapropiado de herramientas de Inteligencia Artificial (IA).
  - En caso de usar IAs para alguna parte de la solución, debe indicar qué herramienta usa en cada caso presentrar el/los *prompts* usados (agreguelos como comentarios siempre).
  - La solución debe entregarse exclusivamente como un *notebook* de Colab. No se revisa en ningún otro formato.
  - El notebook entregado debe tener todos los resultados y gráficos, calculados y a la vista.  También debe ejecutarse completamente con `Ejecutar Todo` sin producir ningún error (verifique antes de entregar).
  - El notebook debe tener explicaciones detalladas para cada paso del procedimiento usando celdas de texto. No debe poner una celda de código sin explicarla. En caso de incluir ecuaciones debe usar $\LaTeX$.
  - No se aceptan códigos en una sola celda con toda o buena parte de la solución (código *spaguetti*). Desarrolle la solución de forma *didáctica*, celda a celda.
  - Todo el código en Python debe satisfacer el estándar [*PEP-8*](https://peps.python.org/pep-0008/). Lea el estándar y póngalo en práctica.
  - **Se bonificarán la originalidad, la creatividad y los cálculos o gráficos adicionales o ilustrativos que realice más allá del objetivo fundamental de la tarea.**

## Solution

Inicialmente se importan las librerias necesarias para el desarrollo del código:

In [1]:
from scipy import constants
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
from plotly.subplots import make_subplots
from plotly.figure_factory import create_quiver

### Functions

Se deben definir la función que represente el problema relativo de los dos cuerpos en este caso se utiliza la presentada en clases:

In [2]:
def rel2bp_fs(t, ys, mu):
  r = ys[:3]
  v = ys[3:]

  dr_dt = v
  dv_dt = -mu/np.linalg.norm(r)**3 * r

  return np.concatenate([dr_dt, dv_dt])

Se especifican los valores del parametro gravitacional estándar $mu$ ($μ$) de marte y su radio:

In [3]:
# Paramétro gravitacional (mu)
G = 6.67408e-20       # En km^3 / (kg s^2)
m_marte = 6.3902e23   # En kg
mu = G * m_marte

# Radio de Marte
R_marte = 3393.149    # En km

Igualente, se definen la condiciones iniciales tanto para la estación espacial como el vehículo tripulado:

In [4]:
y0_estacion =  [11625.240, -4740.578,  0.000, 0.696,  1.707, 0.000]

y0_vehiculo = [2316.794, -2907.198,  0.000, 3.651,  2.456,  0.000]

Then, the times at which the motion occurs are specified in seconds ($s$):

In [5]:
# Tiempo de lanzamiento de la sonda
t_lanzamiento = 3 * 3600                 # En segundos (s)

# Tiempo de llegada al vehículo tripulado
t_llegada = (200 * 60) + t_lanzamiento     # En segundos (s)

Lists of these times are created to facilitate their use in "solve_ivp":

In [6]:
ts_estacion = np.linspace(0, t_lanzamiento, 100)
ts_vehiculo = np.linspace(0, t_llegada, 100)

A list is also created for the *rendezvous* time, i.e., the motion time of the probe:

In [7]:
ts_rendezvous = np.linspace(3*3600, (200 * 60) + 3*3600, 100)

The motion is solved using these times:

1.   Solución para hallar la posición de la estación a las $3 hrs$
2.   Solución para hallar la posición del vehiculo tripulado despues de las $3 hrs$ y $200 min$


In [8]:
# Solución estación
sol_estacion = solve_ivp(rel2bp_fs, [ts_estacion[0],ts_estacion[-1]],
                              y0_estacion,t_eval=ts_estacion, args=(mu,),
                              method='Radau', rtol=1e-13)
sol_estacion

  message: The solver successfully reached the end of the integration interval.
  success: True
   status: 0
        t: [ 0.000e+00  1.091e+02 ...  1.069e+04  1.080e+04]
        y: [[ 1.163e+04  1.170e+04 ...  4.758e+03  4.571e+03]
            [-4.741e+03 -4.554e+03 ...  1.162e+04  1.170e+04]
            ...
            [ 1.707e+00  1.718e+00 ...  6.988e-01  6.714e-01]
            [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]]
      sol: None
 t_events: None
 y_events: None
     nfev: 957
     njev: 2
      nlu: 10

In [9]:
# Solución vehículo tripulado
sol_vehiculo = solve_ivp(rel2bp_fs, [ts_vehiculo[0],ts_vehiculo[-1]],
                              y0_vehiculo,t_eval=ts_vehiculo, args=(mu,),
                              method='Radau', rtol=1e-13)
sol_vehiculo

  message: The solver successfully reached the end of the integration interval.
  success: True
   status: 0
        t: [ 0.000e+00  2.303e+02 ...  2.257e+04  2.280e+04]
        y: [[ 2.317e+03  3.102e+03 ... -1.112e+04 -1.125e+04]
            [-2.907e+03 -2.284e+03 ...  1.637e+04  1.622e+04]
            ...
            [ 2.456e+00  2.932e+00 ... -6.440e-01 -6.647e-01]
            [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]]
      sol: None
 t_events: None
 y_events: None
     nfev: 2906
     njev: 2
      nlu: 36

Se extraen las posiciones de salida y llegada de la estación.

In [10]:
# Salida
posicion_sal_estacion = sol_estacion.y[:3, 0]
posicion_sal_estacion

# Llegada
posicion_llegada_estacion = sol_estacion.y[:3, -1]

posicion_sal_estacion, posicion_llegada_estacion

(array([11625.24 , -4740.578,     0.   ]),
 array([ 4571.02576709, 11697.28008954,     0.        ]))

The departure and arrival positions of the vehicle are extracted.

In [11]:
# Salida
posicion_sal_vehiculo = sol_vehiculo.y[:3, 0]

# Llegada
posicion_llegada_vehiculo = sol_vehiculo.y[:3, -1]

posicion_sal_vehiculo, posicion_llegada_vehiculo

(array([ 2316.794, -2907.198,     0.   ]),
 array([-11246.50790531,  16219.77518218,      0.        ]))

The launch and arrival positions of the probe are extracted:

In [12]:
# Lanzamiento
posicion_lanzamiento = sol_estacion.y[:3, -1]
posicion_lanzamiento

array([ 4571.02576709, 11697.28008954,     0.        ])

In [13]:
# Llegada
posicion_llegada = sol_vehiculo.y[:3, -1]
posicion_llegada

array([-11246.50790531,  16219.77518218,      0.        ])

Ahora, se debe crear una función a modo de determinar la velocidad con la que debe salir la sonda de la estación teniendo en cuenta que, la dintancia entre la sonda y el vehiculo sea cero, es decir que esten en la misma posición.

In [14]:
def error_rendezvous(vel_sonda):
    y0_sonda = np.concatenate((posicion_lanzamiento, vel_sonda))
    sol_sonda = solve_ivp(rel2bp_fs, [ts_rendezvous[0],ts_rendezvous[-1]],
                              y0_sonda,t_eval=ts_rendezvous, args=(mu,),
                              method='Radau', rtol=1e-13)
    sol_sonda
    posicion_sonda = sol_sonda.y[:3, -1]
    return np.linalg.norm(posicion_sonda - posicion_llegada)

Luego, para hallar la velocidad de salida se debe minimizar la distancia lo máximo posible hasta llegar a cero.

Se parte de la velocidad de lanzamiento desde la estación.

In [15]:
vel_lanzamiento = sol_estacion.y[3:, -1]

Therefore, the *minimize* function from *scipy* is used to minimize the objective function.

In [16]:
resultado = minimize(error_rendezvous, vel_lanzamiento, method='BFGS',
                     options={'gtol':1e-13})
vel_salida = resultado.x

*Nota*: Los parametros de la función minimize fueron completados por gemini.

In [17]:
vel_salida

array([-1.29127841e+00,  1.45123188e+00, -2.52472338e-09])

### 2D Plots
Ahora se pueden graficar los moviemtos descritos por cada aeronave.

Pero antes se debe extraer la trayectoria de la sonda con la velocidad óptima.

In [18]:
y0_sonda = np.concatenate((posicion_lanzamiento, vel_salida))
sol_sonda = solve_ivp(rel2bp_fs, [ts_rendezvous[0], ts_rendezvous[-1]],
                      y0_sonda, t_eval=ts_rendezvous, args=(mu,),
                      method='Radau', rtol=1e-13)

In [19]:
vel_llegada_sonda = sol_sonda.y[3:, -1]
vel_llegada_sonda

array([-1.07618971e+00, -3.80785622e-01, -1.25990872e-09])

Now the 2D plots are generated using plotly.

* The figure is created

In [20]:
fig0 = go.Figure()

fig0.update_layout(
    title="Rendezvous maneuver around Mars",
    xaxis_title="x [km]",
    yaxis_title="y [km]",
    showlegend=True,
    width = 800,
    height = 500,
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(scaleanchor='x', scaleratio=1),

)

* Values for plotting Mars.

In [21]:
tetas = np.linspace(0, 2*np.pi, 100)
x_marte = R_marte * np.cos(tetas)
y_marte = R_marte * np.sin(tetas)

* The depiction of Mars is added.

In [22]:
fig0.add_trace(go.Scatter(x=x_marte,
                          y=y_marte,
                          mode='lines',
                          name='Mars',
                          line=dict(color='red')
))

* Se añaden la trayectoria de la estación.

In [23]:
fig0.add_trace(go.Scatter(
    x=sol_estacion.y[0],
    y=sol_estacion.y[1],
    mode='lines',
    name='Station',
    line=dict(color='blue')
))

* Now, the manned vehicle trajectory is added.

In [24]:
fig0.add_trace(go.Scatter(
    x=sol_vehiculo.y[0],
    y=sol_vehiculo.y[1],
    mode='lines',
    name='Manned Vehicle',
    line=dict(color='green')
))

* Then the probe trajectory.

In [25]:
fig0.add_trace(go.Scatter(
    x=sol_sonda.y[0],
    y=sol_sonda.y[1],
    mode='lines',
    name='Probe (rendezvous)',
    line=dict(color='orange')
))

* Finally, the launch and arrival points of the probe

In [26]:
# Launch point
fig0.add_trace(go.Scatter(
    x=[posicion_lanzamiento[0]],
    y=[posicion_lanzamiento[1]],
    mode='markers',
    name='Launch Point',
    marker=dict(color='purple', size=10)
))

# Arrival point
fig0.add_trace(go.Scatter(
    x=[posicion_llegada[0]],
    y=[posicion_llegada[1]],
    mode='markers',
    name='Arrival Point',
    marker=dict(color='purple', size=10)
))

### 3D Graph

The values for plotting Mars in 3D are defined initially.

In [27]:
alpha, beta = np.linspace(0, 2 * np.pi, 100), np.linspace(0, np.pi, 100)
x0 = R_marte * np.outer(np.cos(alpha), np.sin(beta))
y0 = R_marte * np.outer(np.sin(alpha), np.sin(beta))
z0 = R_marte * np.outer(np.ones(np.size(alpha)), np.cos(beta))

The figure is created with the necessary data.

In [28]:
fig1 = go.Figure()

The trajectories are added.

+ For the station.

In [29]:
fig1.add_trace(go.Scatter3d(
    x=sol_estacion.y[0],
    y=sol_estacion.y[1],
    z=sol_estacion.y[2],
    mode='lines',
    name='Station',
    line=dict(color='blue')
))

+ For the manned vehicle.

In [30]:
fig1.add_trace(go.Scatter3d(
    x=sol_vehiculo.y[0],
    y=sol_vehiculo.y[1],
    z=sol_vehiculo.y[2],
    mode='lines',
    name='Manned Vehicle',
    line=dict(color='green')
))

+ For the probe.

In [31]:
fig1.add_trace(go.Scatter3d(
    x=sol_sonda.y[0],
    y=sol_sonda.y[1],
    z=np.zeros_like(tetas),
    mode='lines',
    name='Probe (rendezvous)',
    line=dict(color='orange')
))

+ The depiction of Mars is added.

In [32]:
fig1.add_trace(go.Surface(
    x=x0,
    y=y0,
    z=z0,
    showscale=False,
    colorscale='Reds',
    opacity=0.25
))

+ Finally, the launch and arrival points of the probe

In [33]:
# Launch point
fig1.add_trace(go.Scatter3d(
    x=[posicion_lanzamiento[0]],
    y=[posicion_lanzamiento[1]],
    z=[posicion_lanzamiento[2]],
    mode='markers',
    name='Launch Point',
    marker=dict(color='purple', size=5)
))

# Arrival point
fig1.add_trace(go.Scatter3d(
    x=[posicion_llegada[0]],
    y=[posicion_llegada[1]],
    z=[posicion_llegada[2]],
    mode='markers',
    name='Arrival Point',
    marker=dict(color='purple', size=5)
))

This graph can be animated.


In [34]:
fig2 = go.Figure(
    data=[go.Scatter3d(x=sol_estacion.y[0],
                       y=sol_estacion.y[1],
                       z=sol_estacion.y[2],
                       mode='lines', name='Station',
                       line=dict(color='blue')),
          go.Scatter3d(x=sol_vehiculo.y[0],
                       y=sol_vehiculo.y[1],
                       z=sol_vehiculo.y[2],
                       mode='lines', name='Vehicle',
                       line=dict(color='green')),
          go.Scatter3d(x=sol_sonda.y[0],
                       y=sol_sonda.y[1],
                       z=sol_sonda.y[2],
                       mode='lines', name='Probe',
                       line=dict(color='orange')),
          go.Scatter3d(x=[posicion_lanzamiento[0]],
                       y=[posicion_lanzamiento[1]],
                       z=[posicion_lanzamiento[2]],
                       mode='markers',
                       name='Launch Point',
                       marker=dict(color='purple', size=5)),
          go.Scatter3d(x=[posicion_llegada[0]],
                       y=[posicion_llegada[1]],
                       z=[posicion_llegada[2]],
                       mode='markers',
                       name='Arrival Point',
                       marker=dict(color='purple', size=5))
          ],
    layout=go.Layout(
        title="3D Rendezvous Animation",
        scene=dict(
            xaxis_title="X (km)",
            yaxis_title="Y (km)",
            zaxis_title="Z (km)",
            aspectmode="data"
        ),
        updatemenus=[dict(
            type="buttons",
            buttons=[dict(label="Play",
                          method="animate",
                          args=[None])])]
    ),
    frames=[go.Frame(data=[go.Scatter3d(x=sol_estacion.y[0][:i+1],
                                        y=sol_estacion.y[1][:i+1],
                                        z=sol_estacion.y[2][:i+1],
                                        mode='lines', name='Station',
                                        line=dict(color='blue')),
                         go.Scatter3d(x=sol_vehiculo.y[0][:i+1],
                                      y=sol_vehiculo.y[1][:i+1],
                                      z=sol_vehiculo.y[2][:i+1],
                                      mode='lines', name='Vehicle',
                                      line=dict(color='green')),
                         go.Scatter3d(x=sol_sonda.y[0][:i-1],
                                      y=sol_sonda.y[1][:i-1],
                                      z=sol_sonda.y[2][:i-1],
                                      mode='lines', name='Probe',
                                      line=dict(color='orange')),
                         go.Scatter3d(x=[posicion_lanzamiento[0]],
                                      y=[posicion_lanzamiento[1]],
                                      z=[posicion_lanzamiento[2]],
                                      mode='markers',
                                      name='Launch Point',
                                      marker=dict(color='purple', size=5)),
                         go.Scatter3d(x=[posicion_llegada[0]],
                                      y=[posicion_llegada[1]],
                                      z=[posicion_llegada[2]],
                                      mode='markers',
                                      name='Arrival Point',
                                      marker=dict(color='purple', size=5))])
                                      for i in range(len(sol_estacion.t))]
)

fig2.add_trace(go.Surface(x=x0, y=y0, z=z0, showscale=False, opacity=0.25,
                          colorscale='Reds', name='Marte'))


fig2.show()


### Velocity Results

A DataFrame is created to present the departure and arrival velocity data for the probe.

In [35]:
df_velocidades = pd.DataFrame({
    'Component': ['v_x', 'v_y', 'v_z'],
    'Exit Velocity [km/s]': vel_salida,
    'Arrival Velocity [km/s]': vel_llegada_sonda
})

df_velocidades_T = df_velocidades.set_index('Component').T
print(df_velocidades_T)


Component                     v_x       v_y           v_z
Exit Velocity [km/s]    -1.291278  1.451232 -2.524723e-09
Arrival Velocity [km/s] -1.076190 -0.380786 -1.259909e-09


### Orbital properties ($a, p, q, e, I, \omega, T$)

To calculate the orbital properties of the station, the probe, and the crewed vehicle, the necessary formulas for each property must be applied.


Initially, the key vectors needed to calculate their orbital parameters must be defined.

For the Station:

In [36]:
# Exit
r_estacion_salida = sol_estacion.y[:3, 0]
v_estacion_salida = sol_estacion.y[3:, 0]

# Arrival
r_estacion_llegada = sol_estacion.y[:3, -1]
v_estacion_llegada = sol_estacion.y[3:, -1]

For the probe:

In [37]:
# Exit
r_sonda_salida = sol_sonda.y[:3, 0]
v_sonda_salida = vel_salida
# Arrival
r_sonda_llegada = sol_sonda.y[:3, -1]
v_sonda_llegada = sol_sonda.y[3:, -1]

For the Manned vehicle:

In [38]:
# Exit
r_vehiculo_salida = sol_vehiculo.y[:3, 0]
v_vehiculo_salida = sol_vehiculo.y[3:, 0]

# Arrival
r_vehiculo_llegada = sol_vehiculo.y[:3, -1]
v_vehiculo_llegada = sol_vehiculo.y[3:, -1]

The orbital properties are calculated for one of the bodies; in this case, the probe:

+ Definition of the position and velocity vectors.

In [39]:
r0 = np.array(r_sonda_salida)
v0 = np.array(v_sonda_salida)

+ Their magnitudes are calculated.

In [40]:
r_mag0 = np.linalg.norm(r0)
v_mag0 = np.linalg.norm(v0)

Now the angular momentum and its magnitude are calculated:

$$
Angular \thinspace momentum: \vec h = \vec r \times \dot{\vec r}
$$

In [41]:
h_vec0 = np.cross(r0, v0)
h0 = np.linalg.norm(h_vec0)

With this, the eccentricity can be calculated:

$$
Eccentricity: \vec e = \frac{\dot{\vec r} \times \vec h}{μ} - \frac{\vec r}{r}
$$

In [42]:
e_vec0 = (np.cross(v0, h_vec0) / mu) - (r0 / r_mag0)
e0 = np.linalg.norm(e_vec0)

In [43]:
print(f"The eccentrcity of the orbit is {e0:.4f} ")

The eccentrcity of the orbit is 0.4646 


Knowing the eccentricity, the semi-major axis, semi-minor axis, and perigee of the orbit can be determined:


$$
semi-major \thinspace axis: a = \frac{h^2}{μ} \frac{1}{1-e^2}
$$

$$
semi-minor \thinspace axis: b = a \sqrt{1-e^2}
$$

$$
perigee: q = a(1-e)
$$

In [44]:
# Semi-major axis
a0 = (h0**2) / (mu * (1 - e0**2))

# Semi-minor axis
b0 = a0*(1-e0**2)**0.5

# Perigee distance
q0 = a0 * (1 - e0)

In [45]:
print(f"The semi-major axis of the orbit is {a0:.2f} km ")

The semi-major axis of the orbit is 14129.43 km 


In [46]:
print(f"The semi-minor axis of the orbit is {b0:.2f} km ")

The semi-minor axis of the orbit is 12512.10 km 


In [47]:
print(f"The perigee distance of the orbit is {q0:.2f} km ")

The perigee distance of the orbit is 7565.28 km 


The semilatus rectum can also be calculated:

$$
Semilatus \thinspace rectum: p = \frac{h^2}{μ}
$$

In [48]:
p0 = h0**2 / mu

In [49]:
print(f"The semilatus rectum of the orbit is {p0:.2f} km ")

The semilatus rectum of the orbit is 11079.90 km 


Then the orbital period is calculated:

$$
Orbital \thinspace period: T = \frac{2π}{μ} a^{3/2}
$$

In [50]:
T0 = ((2*np.pi)/(mu**0.5))*(a0**1.5) if e0 < 1 else np.nan

In [74]:
print(f"The orbital period is {T0:.0f} s")

The orbital period is 51099 s


The inclination can also be calculated, but since this is a two-dimensional problem, its value is zero:
$$
Inclination:  cos^{-1}(\frac{hz}{| \vec h|})
$$


In [75]:
I0 = np.arccos(h_vec0[2] / h0)

In [76]:
print(f"The inclination is {I0:.2f} rad")

The inclination is 0.00 rad


Finally, the argument of periapsis is calculated with this formula:
$$
Argument \thinspace of \thinspace periapsis:  cos^{-1}(\frac{\vec n \cdot \vec e}{|\vec n||\vec e|})
$$

$$
\vec n = \vec k \times \vec h
$$

But since it is not a three-dimensional problem, it reduces to:
$$
ω = arctan2(e_y,e_x)
$$


In [54]:
omega0 = np.arctan2(e_vec0[1], e_vec0[0])

In [77]:
print(f"The argument of the periapsis is {omega0:.2f} rad")

The argument of the periapsis is -0.63 rad


Since the above procedure can become a routine, the following function is defined.

In [80]:
def prop_orbitales(r_vec, v_vec, mu):
  r = np.array(r_vec)
  v = np.array(v_vec)

  # Magnitudes
  r_mag = np.linalg.norm(r)
  v_mag = np.linalg.norm(v)

  # Angular momentum
  h_vec = np.cross(r, v)
  h = np.linalg.norm(h_vec)

  # Eccentricity
  e_vec = (np.cross(v, h_vec) / mu) - (r / r_mag)
  e = np.linalg.norm(e_vec)

  # Semi-major axis
  a = (h**2) / (mu * (1 - e**2))

  # Semi-minor axis
  b = a*(1-e**2)**0.5

  # Pericenter
  q = a * (1 - e)

  # Semilatus rectum
  p = h**2 / mu

  # Orbital period
  T = ((2*np.pi)/(mu**0.5))*(a**1.5) if e < 1 else np.nan

  # Inclination
  I = np.arccos(h_vec[2] / h)

  # Argument of the periapsis
  omega = np.arctan2(e_vec[1], e_vec[0])


  return {
        "Eccentricity e": e,
        "Semi-major axis a (km)": a,
        "Semi-minor axis b (km)": b,
        "Pericenter q (km)": q,
        "Semilatus rectum p (km)": p,
        "Orbital period T (s)": T,
        "Inclination I (rad)": I,
        "Argument of the periapsis omega (rad)": omega
  }

With this, it is now possible to extract the orbital properties. This can be done with the departure or arrival data, since the same result would be obtained:



*   For the station:



In [81]:
prop_estacion = prop_orbitales(r_estacion_salida, v_estacion_salida, mu)



*   For the probe:

In [82]:
prop_sonda = prop_orbitales(r_sonda_salida, v_sonda_salida, mu)

* For the manned vehicle:

In [83]:
prop_vehiculo = prop_orbitales(r_vehiculo_salida, v_vehiculo_salida, mu)

A DataFrame is created to store this data.



In [84]:
df_orbitas_0 = pd.DataFrame([prop_estacion, prop_sonda, prop_vehiculo])
df_orbitas_0.index = ['Estación', 'Sonda', 'Vehículo tripulado']
df_orbitas_0

,Eccentricity e,Semi-major axis a (km),Semi-minor axis b (km),Pericenter q (km),Semilatus rectum p (km),Orbital period T (s),Inclination I (rad),Argument of the periapsis omega (rad)
Estación,0.000362,12559.165757,12559.164934,12554.619443,12559.164111,42822.116370,0.0,-0.267554
Sonda,0.464573,14129.429244,12512.100819,7565.276301,11079.900271,51099.172509,0.0,-0.628784
Vehículo tripulado,0.690135,11901.456561,8612.858535,3687.849794,6232.962475,39502.730203,0.0,-1.095701


### Vector graph



Furthermore, using these values, it is now possible to plot the velocity vectors at the probe for the 2D graph

To do this, it's necessary to find the angle $\theta$ at which it is in both the launch position and the arrival position.

By definition of the scalar product.

$$
\vec a \cdot \vec b = |\vec a| \cdot |\vec b| cosθ
$$

Therefore, $\theta$ is equal to:

$$
θ = arccos(\frac{\vec a \cdot \vec b }{|\vec a| \cdot |\vec b|})
$$


+ For the exit.

In [61]:
teta_0 = np.arccos(np.dot(r_sonda_salida, v_sonda_salida) /
 (np.linalg.norm(r_sonda_salida) * np.linalg.norm(v_sonda_salida)))

teta_0

np.float64(1.0996705666029025)

+ For the arrival.

In [62]:
teta_f = np.arccos(np.dot(r_sonda_llegada, v_sonda_llegada) /
 (np.linalg.norm(r_sonda_llegada) * np.linalg.norm(v_sonda_llegada)))

teta_f

np.float64(1.3046060970864768)

The radial unit vector is calculated.

In [63]:
# Salida
ur_x_0 = np.cos(teta_0)
ur_y_0 = np.sin(teta_0)

# Llegada
ur_x_f = np.cos(teta_f)
ur_y_f = np.sin(teta_f)

Tangential unit vector.


In [64]:
#Salida
ut_x_0 = -np.sin(teta_0)
ut_y_0 = np.cos(teta_0)

# Llegada
ut_x_f = -np.sin(teta_f)
ut_y_f = np.cos(teta_f)

Radial velocity.

In [65]:
# Sálida
vr_0 = mu/h0 * e0 * np.sin(teta_0)
vr_x_0 = vr_0 * ur_x_0
vr_y_0 = vr_0 * ur_y_0

# Llegada
vr_f = mu/h0 * e0 * np.sin(teta_f)
vr_x_f = vr_f * ur_x_f
vr_y_f = vr_f * ur_y_f

Tangential velocity.

In [66]:
# Salida
vt_0 = mu/h0 * (1 + e0*np.cos(teta_0))
vt_x_0 = vt_0 * ut_x_0
vt_y_0 = vt_0 * ut_y_0

# Llegada
vt_f = mu/h0 * (1 + e0*np.cos(teta_f))
vt_x_f = vt_f * ut_x_f
vt_y_f = vt_f * ut_y_f

Total velocity

In [67]:
# Salida
v_x_0 = vr_x_0 + vt_x_0
v_y_0 = vr_y_0 + vt_y_0

# Llegada
v_x_f = vr_x_f + vt_x_f
v_y_f = vr_y_f + vt_y_f

Then, the probe velocity vectors are added to the 2D plot created earlier.

+ Total velocity

In [68]:
vector = create_quiver(
    x=[posicion_lanzamiento[0], posicion_llegada[0]],
    y=[posicion_lanzamiento[1], posicion_llegada[1]],
    u=[v_x_0, v_x_f],
    v=[v_y_0, v_y_f],
    scale=1000,
    line=dict(color='black'),
    name='Velocidad',
    showlegend=False,
)

for trace in vector.data:

    fig0.add_trace(trace)
fig0.show()

+ Tangential velocity

In [69]:
vector_tan = create_quiver(
    x=[posicion_lanzamiento[0], posicion_llegada[0]],
    y=[posicion_lanzamiento[1], posicion_llegada[1]],
    u=[vt_x_0, vt_x_f],
    v=[vt_y_0, vt_y_f],
    scale=1000,
    line=dict(color='pink'),
    name='Velocidad',
    showlegend=False,
)

for trace_tan in vector_tan.data:

    fig0.add_trace(trace_tan)
fig0.show()

+ Radial velocity

In [70]:
vector_rad = create_quiver(
    x=[posicion_lanzamiento[0], posicion_llegada[0]],
    y=[posicion_lanzamiento[1], posicion_llegada[1]],
    u=[vr_x_0, vr_x_f],
    v=[vr_y_0, vr_y_f],
    scale=1000,
    line=dict(color='red'),
    name='Velocidad',
    showlegend=False,
)

for trace_rad in vector_rad.data:

    fig0.add_trace(trace_rad)
fig0.show()

It is also possible to plot the flight path angle of the probe at the points mentioned above.

$$
γ = arctan(\frac{esinθ}{1 + ecosθ})
$$

In [85]:
# Exit
gamma_0 = np.arctan(e0*np.sin(teta_0)/(1+e0*np.cos(teta_0))) * 180/np.pi

# Arrival
gamma_f = np.arctan(e0*np.sin(teta_f)/(1+e0*np.cos(teta_f))) * 180/np.pi

In [72]:
angulo = fig0.add_annotation(
    x=posicion_lanzamiento[0], y=posicion_lanzamiento[1],
    text=f"$\gamma = {gamma_0:.1f}^\circ$",
    xshift=0,yshift=0,
    xanchor='left'
)
fig0.show()

<>:3: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<>:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:3: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<>:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\User\AppData\Local\Temp\ipykernel_10652\3939326961.py:3: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
  text=f"$\gamma = {gamma_0:.1f}^\circ$",
C:\Users\User\AppData\Local\Temp\ipykernel_10652\3939326961.py:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences w

In [73]:
angulo = fig0.add_annotation(
    x=posicion_llegada[0], y=posicion_llegada[1],
    text=f"$\gamma = {gamma_f:.1f}^\circ$",
    xshift=0,yshift=0,
    xanchor='left'
)
fig0.show()

<>:3: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<>:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:3: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<>:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\User\AppData\Local\Temp\ipykernel_10652\516485598.py:3: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
  text=f"$\gamma = {gamma_f:.1f}^\circ$",
C:\Users\User\AppData\Local\Temp\ipykernel_10652\516485598.py:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences wil